# Coastal Water-Level Risk Analysis in Python

This notebook demonstrates an engineering-style analysis of coastal water-level time-series data.

The project uses a synthetic but realistic dataset containing:
- tidal variation,
- seasonal variation,
- random noise,
- storm-surge events,
- a small long-term sea-level trend.

The workflow is intended to be transferable to real measured tide-gauge data.

## 1. Import libraries and define project paths

We use a small, practical Python stack: `pandas`, `NumPy`, `matplotlib`, and `pathlib`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make plots readable and consistent
plt.rcParams.update({
    "figure.figsize": (12, 5),
    "axes.grid": True,
    "font.size": 11,
})

# Define project folders
PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
FIGURES_DIR = PROJECT_ROOT / "figures"

DATA_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

DATA_PATH = DATA_DIR / "synthetic_water_level.csv"
FLOOD_THRESHOLD_M = 1.50

## 2. Generate synthetic coastal water-level data

The generated signal contains astronomical tide, seasonal variation, noise, storm surges, and a small sea-level rise trend.

The dataset is synthetic, but the same analysis structure can be applied to observed tide-gauge measurements.

In [ ]:
# The generator is stored in the src folder
import sys
sys.path.append(str(PROJECT_ROOT / "src"))

from generate_synthetic_water_level import generate_synthetic_water_level

df = generate_synthetic_water_level(output_path=DATA_PATH)
df.head()

## 3. Load and inspect the dataset

In a real project, this step would include checking timestamps, missing values, duplicate records, sensor units, and suspicious values.

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

print("Dataset shape:", df.shape)
print("Time range:", df["timestamp"].min(), "to", df["timestamp"].max())
print("Missing values:")
print(df.isna().sum())

df.describe()

## 4. Plot the full water-level time series

The threshold line represents a simplified flood-risk screening level.
Points above this level are treated as potential exceedance conditions.

In [ ]:
fig, ax = plt.subplots()

ax.plot(df["timestamp"], df["water_level_m"], linewidth=0.8, label="Synthetic water level")
ax.axhline(FLOOD_THRESHOLD_M, linestyle="--", linewidth=2, label=f"Flood threshold = {FLOOD_THRESHOLD_M:.2f} m")

ax.set_title("Synthetic Coastal Water-Level Time Series")
ax.set_xlabel("Time")
ax.set_ylabel("Water level [m above reference]")
ax.legend()

fig.tight_layout()
fig.savefig(FIGURES_DIR / "water_level_timeseries.png", dpi=200)
plt.show()

## 5. Detect threshold exceedance points

An exceedance point is any timestamp where the water level is above the selected flood-risk threshold.

In [ ]:
df["exceeds_threshold"] = df["water_level_m"] > FLOOD_THRESHOLD_M

total_exceedance_hours = int(df["exceeds_threshold"].sum())
percentage_exceedance = 100 * df["exceeds_threshold"].mean()

print(f"Total time above threshold: {total_exceedance_hours} hours")
print(f"Share of record above threshold: {percentage_exceedance:.2f}%")

## 6. Group consecutive exceedance points into events

Instead of counting every hourly point separately, consecutive exceedance timestamps are grouped into storm/flood events.

This is important because one storm can create many consecutive exceedance points.

In [ ]:
# Create a new event ID whenever the exceedance status changes
df["exceedance_group"] = (df["exceeds_threshold"] != df["exceeds_threshold"].shift()).cumsum()

events = (
    df[df["exceeds_threshold"]]
    .groupby("exceedance_group")
    .agg(
        start_time=("timestamp", "min"),
        end_time=("timestamp", "max"),
        duration_hours=("timestamp", "count"),
        peak_water_level_m=("water_level_m", "max"),
        mean_water_level_m=("water_level_m", "mean"),
    )
    .reset_index(drop=True)
)

events["exceedance_height_m"] = events["peak_water_level_m"] - FLOOD_THRESHOLD_M
events = events.sort_values("peak_water_level_m", ascending=False).reset_index(drop=True)

events

## 7. Engineering summary statistics

These indicators provide a quick flood-risk screening summary:
- number of events,
- total time above threshold,
- maximum observed water level,
- longest event,
- highest exceedance above threshold.

In [ ]:
summary = {
    "number_of_exceedance_events": len(events),
    "total_time_above_threshold_hours": int(events["duration_hours"].sum()),
    "maximum_water_level_m": df["water_level_m"].max(),
    "longest_event_hours": events["duration_hours"].max(),
    "maximum_exceedance_height_m": events["exceedance_height_m"].max(),
}

summary_df = pd.DataFrame(summary, index=["value"]).T
summary_df

## 8. Highlight exceedance events on the time series

This plot makes it easy to identify when flood-risk conditions occurred.

In [ ]:
fig, ax = plt.subplots()

ax.plot(df["timestamp"], df["water_level_m"], linewidth=0.8, label="Water level")
ax.scatter(
    df.loc[df["exceeds_threshold"], "timestamp"],
    df.loc[df["exceeds_threshold"], "water_level_m"],
    s=14,
    label="Exceedance points",
)
ax.axhline(FLOOD_THRESHOLD_M, linestyle="--", linewidth=2, label=f"Threshold = {FLOOD_THRESHOLD_M:.2f} m")

ax.set_title("Detected Threshold Exceedance Events")
ax.set_xlabel("Time")
ax.set_ylabel("Water level [m above reference]")
ax.legend()

fig.tight_layout()
fig.savefig(FIGURES_DIR / "exceedance_events.png", dpi=200)
plt.show()

## 9. Water-level distribution

The histogram shows how common normal water levels are compared with high-water extremes.

In [ ]:
fig, ax = plt.subplots()

ax.hist(df["water_level_m"], bins=60, edgecolor="black")
ax.axvline(FLOOD_THRESHOLD_M, linestyle="--", linewidth=2, label=f"Threshold = {FLOOD_THRESHOLD_M:.2f} m")

ax.set_title("Distribution of Synthetic Coastal Water Levels")
ax.set_xlabel("Water level [m above reference]")
ax.set_ylabel("Frequency [hours]")
ax.legend()

fig.tight_layout()
fig.savefig(FIGURES_DIR / "water_level_histogram.png", dpi=200)
plt.show()

## 10. Monthly exceedance counts

Monthly exceedance counts help identify whether high-water risk is concentrated in particular seasons.

In [ ]:
df["month"] = df["timestamp"].dt.to_period("M").dt.to_timestamp()

monthly_exceedances = (
    df[df["exceeds_threshold"]]
    .groupby("month")
    .size()
    .rename("exceedance_hours")
    .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 5))

ax.bar(monthly_exceedances["month"], monthly_exceedances["exceedance_hours"], width=20)

ax.set_title("Monthly Time Above Flood-Risk Threshold")
ax.set_xlabel("Month")
ax.set_ylabel("Exceedance duration [hours]")

fig.tight_layout()
fig.savefig(FIGURES_DIR / "monthly_exceedances.png", dpi=200)
plt.show()

monthly_exceedances

## 11. Simplified return-period / ranked extreme-value plot

This is not a full extreme-value analysis.  
It is a simple ranking of annual maxima using the Weibull plotting-position formula:

`return period = (n + 1) / rank`

where rank 1 is the largest annual maximum.

For real design work, longer measured records and formal extreme-value methods would be required.

In [ ]:
annual_maxima = (
    df.set_index("timestamp")["water_level_m"]
    .resample("YE")
    .max()
    .rename("annual_max_water_level_m")
    .reset_index()
)

annual_maxima["year"] = annual_maxima["timestamp"].dt.year
annual_maxima = annual_maxima.sort_values("annual_max_water_level_m", ascending=False).reset_index(drop=True)

n_years = len(annual_maxima)
annual_maxima["rank"] = np.arange(1, n_years + 1)
annual_maxima["return_period_years"] = (n_years + 1) / annual_maxima["rank"]

fig, ax = plt.subplots()

ax.plot(
    annual_maxima["return_period_years"],
    annual_maxima["annual_max_water_level_m"],
    marker="o",
    linewidth=1.5,
)
ax.axhline(FLOOD_THRESHOLD_M, linestyle="--", linewidth=2, label=f"Threshold = {FLOOD_THRESHOLD_M:.2f} m")

ax.set_title("Simplified Ranked Annual-Maximum Water Levels")
ax.set_xlabel("Return period estimate [years]")
ax.set_ylabel("Annual maximum water level [m]")
ax.legend()

fig.tight_layout()
fig.savefig(FIGURES_DIR / "return_period_curve.png", dpi=200)
plt.show()

annual_maxima

## 12. Event duration versus peak water level

This plot compares event severity using two simple indicators:
- event duration,
- maximum water level reached during the event.

Events that are both long and high are generally more critical for flood-risk screening.

In [ ]:
fig, ax = plt.subplots()

ax.scatter(events["duration_hours"], events["peak_water_level_m"], s=80)

for i, row in events.iterrows():
    ax.annotate(
        str(i + 1),
        (row["duration_hours"], row["peak_water_level_m"]),
        textcoords="offset points",
        xytext=(5, 5),
    )

ax.axhline(FLOOD_THRESHOLD_M, linestyle="--", linewidth=2, label=f"Threshold = {FLOOD_THRESHOLD_M:.2f} m")

ax.set_title("Exceedance Event Duration vs Peak Water Level")
ax.set_xlabel("Event duration [hours]")
ax.set_ylabel("Peak water level [m]")
ax.legend()

fig.tight_layout()
fig.savefig(FIGURES_DIR / "event_duration_vs_peak.png", dpi=200)
plt.show()

## 13. Export event statistics

The detected event table can be exported and used in reports, dashboards, or further analysis.

In [ ]:
events_path = DATA_DIR / "detected_exceedance_events.csv"
events.to_csv(events_path, index=False)

print(f"Event table saved to: {events_path}")
events

## 14. Engineering interpretation

The synthetic dataset produced several threshold exceedance events over the two-year period.

The analysis shows how a coastal water-level screening workflow can:
- identify high-water periods,
- group exceedance points into physically meaningful events,
- estimate event duration and peak severity,
- compare seasonal exceedance patterns,
- communicate results using clear engineering plots.

For real coastal flood-risk assessment, the next steps would be:
- use measured tide-gauge or modelled water-level data,
- validate datum/reference levels,
- apply site-specific flood thresholds,
- use longer records for extreme-value analysis,
- include waves, wind setup, river discharge, and local topography where relevant.